# 18vA — Residual-model design and leakage-safe feature panels

This stage freezes the candidate residual models, the low-dimensional
weather features and the common predictive quantile grid before any
development score is examined.

The target is the deterministic forecast residual

\[
R_{d,r}
=
T_d^{\mathrm{HKO}}
-
\widehat T_{d,r}^{\mathrm{det}}.
\]

The candidate set contains:

- the uncorrected deterministic forecast;
- pooled and rule-specific mean-residual corrections;
- pooled and rule-specific empirical residual distributions;
- rule-specific Gaussian processes with RBF and Matérn-\(3/2\)
  kernels;
- pooled and rule-specific CatBoost quantile regressions.

The Gaussian-process input is one-dimensional time within each
decision rule. The tree features are deterministic forecast
information, calendar position and forecast-run timing only.
Market prices, contract probabilities, the archived Gaussian bridge
and artificial ensemble features are excluded.

Development labels are placed in a separate labelled panel.
Holdout and June panels are blind and contain no realised HKO value,
event outcome or realised residual.

**Revision v2.** The execution is divided into separate baseline/GP and tree OOF notebooks. Tree candidates use one CatBoost `MultiQuantile` fit per scope-fold, and each GP uses one deterministic marginal-likelihood optimisation.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / ".git").exists():
    raise RuntimeError(
        f"Run this notebook from the repository root, not {ROOT}"
    )

UTC = timezone.utc
STEP = "18vA"
SAMPLE_ORIGIN = pd.Timestamp("2026-03-16")
CONTRACTS_PER_BOOK = 11

RULES = [
    "24h_prior",
    "12h_prior",
    "6h_prior",
    "event_day_open",
]
RULE_ORDER = {
    rule: position
    for position, rule in enumerate(RULES)
}

S_DIR = (
    ROOT
    / "data/processed/18s_expanded_march_june_canonical_sample"
)
U_A_DIR = (
    ROOT
    / "data/processed/18uA_chronological_partition_and_folds"
)
U_B_DIR = (
    ROOT
    / "data/processed/18uB_model_specific_freeze_support"
)

WEATHER_PATH = (
    S_DIR / "18s_expanded_deterministic_weather_panel.csv"
)
S_SUMMARY_PATH = S_DIR / "18s_expanded_sample_summary.json"
S_MANIFEST_PATH = S_DIR / "18s_expanded_sha256_manifest.csv"

ASSIGNMENT_PATH = U_A_DIR / "18uA_date_rule_assignment.csv"
U_A_SUMMARY_PATH = U_A_DIR / "18uA_summary.json"
U_A_MANIFEST_PATH = U_A_DIR / "18uA_sha256_manifest.csv"

SCOPE_PATH = U_B_DIR / "18uB_estimator_scope_registry.csv"
FOLD_TRAIN_PATH = (
    U_B_DIR / "18uB_fold_training_membership.csv"
)
FOLD_VALIDATION_PATH = (
    U_B_DIR / "18uB_fold_validation_membership.csv"
)
FINAL_FIT_PATH = U_B_DIR / "18uB_final_fit_membership.csv"
EVALUATION_PATH = U_B_DIR / "18uB_evaluation_support.csv"
U_B_SUMMARY_PATH = U_B_DIR / "18uB_summary.json"
U_B_MANIFEST_PATH = U_B_DIR / "18uB_sha256_manifest.csv"

OUT_DIR = (
    ROOT
    / "data/processed/18vA_residual_model_design_and_features"
)
REPORT_DIR = (
    ROOT
    / "reports/18vA_residual_model_design_and_features"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_INPUTS = [
    WEATHER_PATH,
    S_SUMMARY_PATH,
    S_MANIFEST_PATH,
    ASSIGNMENT_PATH,
    U_A_SUMMARY_PATH,
    U_A_MANIFEST_PATH,
    SCOPE_PATH,
    FOLD_TRAIN_PATH,
    FOLD_VALIDATION_PATH,
    FINAL_FIT_PATH,
    EVALUATION_PATH,
    U_B_SUMMARY_PATH,
    U_B_MANIFEST_PATH,
]

for path in REQUIRED_INPUTS:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required verified input is missing: {path}"
        )

QUANTILE_LEVELS = np.round(
    np.arange(0.01, 1.00, 0.01),
    2,
)
TREE_TRAINING_QUANTILES = [
    0.05,
    0.25,
    0.50,
    0.75,
    0.95,
]

CANDIDATE_ROWS = [
    {
        "candidate_id": "raw_deterministic",
        "model_family": "RAW",
        "scope_type": "POOLED",
        "scope_id_pattern": "pooled_all_rules",
        "distribution_type": "DEGENERATE",
        "complexity_rank": 1,
        "principal_family_comparison": False,
    },
    {
        "candidate_id": "pooled_mean_residual",
        "model_family": "BASELINE",
        "scope_type": "POOLED",
        "scope_id_pattern": "pooled_all_rules",
        "distribution_type": "DEGENERATE",
        "complexity_rank": 2,
        "principal_family_comparison": False,
    },
    {
        "candidate_id": "rule_mean_residual",
        "model_family": "BASELINE",
        "scope_type": "RULE_SPECIFIC",
        "scope_id_pattern": "rule_specific_{decision_rule}",
        "distribution_type": "DEGENERATE",
        "complexity_rank": 3,
        "principal_family_comparison": False,
    },
    {
        "candidate_id": "pooled_empirical_residual",
        "model_family": "BASELINE",
        "scope_type": "POOLED",
        "scope_id_pattern": "pooled_all_rules",
        "distribution_type": "EMPIRICAL_QUANTILES",
        "complexity_rank": 4,
        "principal_family_comparison": False,
    },
    {
        "candidate_id": "rule_empirical_residual",
        "model_family": "BASELINE",
        "scope_type": "RULE_SPECIFIC",
        "scope_id_pattern": "rule_specific_{decision_rule}",
        "distribution_type": "EMPIRICAL_QUANTILES",
        "complexity_rank": 5,
        "principal_family_comparison": False,
    },
    {
        "candidate_id": "gp_rbf_rule",
        "model_family": "GAUSSIAN_PROCESS",
        "scope_type": "RULE_SPECIFIC",
        "scope_id_pattern": "rule_specific_{decision_rule}",
        "distribution_type": "GAUSSIAN_QUANTILES",
        "complexity_rank": 6,
        "principal_family_comparison": True,
    },
    {
        "candidate_id": "gp_matern32_rule",
        "model_family": "GAUSSIAN_PROCESS",
        "scope_type": "RULE_SPECIFIC",
        "scope_id_pattern": "rule_specific_{decision_rule}",
        "distribution_type": "GAUSSIAN_QUANTILES",
        "complexity_rank": 7,
        "principal_family_comparison": True,
    },
    {
        "candidate_id": "catboost_quantile_pooled",
        "model_family": "TREE",
        "scope_type": "POOLED",
        "scope_id_pattern": "pooled_all_rules",
        "distribution_type": "INTERPOLATED_QUANTILES",
        "complexity_rank": 8,
        "principal_family_comparison": True,
    },
    {
        "candidate_id": "catboost_quantile_rule",
        "model_family": "TREE",
        "scope_type": "RULE_SPECIFIC",
        "scope_id_pattern": "rule_specific_{decision_rule}",
        "distribution_type": "INTERPOLATED_QUANTILES",
        "complexity_rank": 9,
        "principal_family_comparison": True,
    },
]

FEATURE_ROWS = [
    {
        "feature_name": "day_index",
        "gp_rule_specific": True,
        "tree_pooled": True,
        "tree_rule_specific": True,
        "description": (
            "Calendar days since 16 March 2026."
        ),
    },
    {
        "feature_name": "forecast_daily_max_c",
        "gp_rule_specific": False,
        "tree_pooled": True,
        "tree_rule_specific": True,
        "description": (
            "Deterministic ECMWF forecast daily maximum."
        ),
    },
    {
        "feature_name": "decision_rule_order",
        "gp_rule_specific": False,
        "tree_pooled": True,
        "tree_rule_specific": False,
        "description": (
            "Ordinal decision-rule indicator for pooled trees."
        ),
    },
    {
        "feature_name": "forecast_lead_hours",
        "gp_rule_specific": False,
        "tree_pooled": True,
        "tree_rule_specific": True,
        "description": (
            "Hours from selected-run availability to decision cut-off."
        ),
    },
    {
        "feature_name": "run_initialisation_hour_utc",
        "gp_rule_specific": False,
        "tree_pooled": True,
        "tree_rule_specific": True,
        "description": (
            "UTC cycle hour of the selected deterministic run."
        ),
    },
    {
        "feature_name": "day_of_year_sin",
        "gp_rule_specific": False,
        "tree_pooled": True,
        "tree_rule_specific": True,
        "description": "Seasonal sine coordinate.",
    },
    {
        "feature_name": "day_of_year_cos",
        "gp_rule_specific": False,
        "tree_pooled": True,
        "tree_rule_specific": True,
        "description": "Seasonal cosine coordinate.",
    },
]

SELECTION_PROTOCOL = {
    "primary_score": "date_balanced_mean_crps_c",
    "secondary_score": "date_balanced_mean_absolute_error_c",
    "tertiary_score": "complexity_rank",
    "common_support_required": True,
    "support_rule": (
        "intersection of temporal OOF predictions and "
        "model-specific freeze-admissible development labels "
        "across all nine candidates"
    ),
    "holdout_used": False,
    "external_test_used": False,
    "family_winners_required": [
        "BASELINE",
        "GAUSSIAN_PROCESS",
        "TREE",
    ],
}

In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


def parse_bool(
    series: pd.Series,
    *,
    name: str,
) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    parsed = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "yes": True,
                "no": False,
            }
        )
    )

    if parsed.isna().any():
        bad = series.loc[
            parsed.isna()
        ].drop_duplicates().tolist()
        raise ValueError(
            f"Could not parse Boolean column {name}: {bad}"
        )

    return parsed.astype(bool)


def verify_manifest(path: Path) -> None:
    manifest = pd.read_csv(path)
    failures: list[str] = []

    for row in manifest.itertuples(index=False):
        candidate = ROOT / row.path

        if not candidate.is_file():
            failures.append(f"MISSING: {row.path}")
            continue

        digest = sha256_file(candidate)
        if digest != row.sha256:
            failures.append(f"HASH: {row.path}")

        if candidate.stat().st_size != int(row.size_bytes):
            failures.append(f"SIZE: {row.path}")

    if failures:
        raise AssertionError(
            f"Manifest verification failed for {path}:\n"
            + "\n".join(failures)
        )


def date_balanced_weights(
    frame: pd.DataFrame,
    date_column: str,
) -> pd.Series:
    n_dates = frame[date_column].nunique()
    rows_per_date = frame.groupby(
        date_column
    )[date_column].transform("size")
    weights = 1.0 / (
        float(n_dates)
        * rows_per_date.astype(float)
    )

    if not np.isclose(
        weights.sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ):
        raise AssertionError(
            "Date-balanced weights do not sum to one."
        )

    return weights


for manifest_path in [
    S_MANIFEST_PATH,
    U_A_MANIFEST_PATH,
    U_B_MANIFEST_PATH,
]:
    verify_manifest(manifest_path)

summaries = {}
for name, path in [
    ("18s", S_SUMMARY_PATH),
    ("18uA", U_A_SUMMARY_PATH),
    ("18uB", U_B_SUMMARY_PATH),
]:
    with path.open(encoding="utf-8") as handle:
        summaries[name] = json.load(handle)

    if summaries[name].get("verdict") != "PASS":
        raise AssertionError(
            f"{name} is not a PASS release."
        )

weather = pd.read_csv(
    WEATHER_PATH,
    dtype={"market_id": str},
    low_memory=False,
)
assignment = pd.read_csv(
    ASSIGNMENT_PATH,
    low_memory=False,
)
scopes = pd.read_csv(
    SCOPE_PATH,
    low_memory=False,
)
fold_train = pd.read_csv(
    FOLD_TRAIN_PATH,
    low_memory=False,
)
fold_validation = pd.read_csv(
    FOLD_VALIDATION_PATH,
    low_memory=False,
)
final_fit = pd.read_csv(
    FINAL_FIT_PATH,
    low_memory=False,
)
evaluation = pd.read_csv(
    EVALUATION_PATH,
    low_memory=False,
)

for frame in [
    weather,
    assignment,
    fold_train,
    fold_validation,
    final_fit,
    evaluation,
]:
    if "event_date" in frame.columns:
        frame["event_date"] = pd.to_datetime(
            frame["event_date"],
            errors="raise",
        )

for frame in [
    weather,
    assignment,
    fold_train,
    fold_validation,
    final_fit,
    evaluation,
]:
    for column in [
        "decision_cutoff_utc",
        "selected_run_initialisation_utc",
        "selected_run_available_utc",
        "current_label_available_utc",
        "model_specific_holdout_freeze_utc",
        "fold_freeze_utc",
    ]:
        if column in frame.columns:
            frame[column] = pd.to_datetime(
                frame[column],
                utc=True,
                errors="raise",
            )

assignment["model_ready"] = parse_bool(
    assignment["model_ready"],
    name="assignment.model_ready",
)

if len(weather) != 4125:
    raise AssertionError(
        f"Expected 4,125 weather contract rows, "
        f"found {len(weather)}."
    )
if len(assignment) != 412:
    raise AssertionError(
        f"Expected 412 assignment rows, found {len(assignment)}."
    )
if len(scopes) != 5:
    raise AssertionError(
        f"Expected five estimator scopes, found {len(scopes)}."
    )

print("Verified 18s and 18u inputs and manifests: PASS")

Verified 18s and 18u inputs and manifests: PASS


In [3]:
path_group = [
    "event_date",
    "decision_rule",
    "decision_rule_order",
    "sample_block",
]

path_checks = (
    weather.groupby(
        path_group,
        as_index=False,
    )
    .agg(
        contract_rows=("market_id", "size"),
        forecast_nunique=(
            "forecast_daily_max_c",
            "nunique",
        ),
        hko_nunique=("hko_daily_max_c", "nunique"),
        cutoff_nunique=(
            "decision_cutoff_utc",
            "nunique",
        ),
        run_init_nunique=(
            "selected_run_initialisation_utc",
            "nunique",
        ),
        run_available_nunique=(
            "selected_run_available_utc",
            "nunique",
        ),
    )
)

if not path_checks["contract_rows"].eq(
    CONTRACTS_PER_BOOK
).all():
    raise AssertionError(
        "A weather path is not an eleven-contract book."
    )

for column in [
    "forecast_nunique",
    "hko_nunique",
    "cutoff_nunique",
    "run_init_nunique",
    "run_available_nunique",
]:
    if not path_checks[column].eq(1).all():
        raise AssertionError(
            f"Inconsistent weather-path field: {column}"
        )

current_paths = (
    weather.sort_values(
        [
            "event_date",
            "decision_rule_order",
            "market_id",
        ]
    )
    .drop_duplicates(
        ["event_date", "decision_rule"]
    )
    [
        [
            "event_date",
            "sample_block",
            "decision_rule",
            "decision_rule_order",
            "decision_cutoff_utc",
            "selected_run_initialisation_utc",
            "selected_run_available_utc",
            "selected_run_key",
            "forecast_daily_max_c",
            "hko_daily_max_c",
            "forecast_error_c",
        ]
    ]
    .copy()
)

current_paths["residual_c"] = (
    current_paths["hko_daily_max_c"]
    - current_paths["forecast_daily_max_c"]
)
current_paths["day_index"] = (
    current_paths["event_date"] - SAMPLE_ORIGIN
).dt.days.astype(float)
current_paths["forecast_lead_hours"] = (
    current_paths["decision_cutoff_utc"]
    - current_paths["selected_run_available_utc"]
).dt.total_seconds() / 3600.0
current_paths["run_initialisation_hour_utc"] = (
    current_paths[
        "selected_run_initialisation_utc"
    ].dt.hour.astype(float)
)
day_of_year = current_paths[
    "event_date"
].dt.dayofyear.astype(float)
current_paths["day_of_year_sin"] = np.sin(
    2.0 * np.pi * day_of_year / 366.0
)
current_paths["day_of_year_cos"] = np.cos(
    2.0 * np.pi * day_of_year / 366.0
)

feature_panel = assignment.merge(
    current_paths,
    on=[
        "event_date",
        "sample_block",
        "decision_rule",
        "decision_rule_order",
        "decision_cutoff_utc",
    ],
    how="left",
    validate="one_to_one",
    suffixes=("", "_weather"),
)

ready = feature_panel.loc[
    feature_panel["model_ready"]
].copy()

development = ready.loc[
    ready["evaluation_block"].eq("DEVELOPMENT")
].copy()
holdout = ready.loc[
    ready["evaluation_block"].eq(
        "INTERNAL_HOLDOUT"
    )
].copy()
external = ready.loc[
    ready["evaluation_block"].eq("EXTERNAL_TEST")
].copy()

if len(development) != 144:
    raise AssertionError(
        f"Expected 144 development rows, found {len(development)}."
    )
if len(holdout) != 40:
    raise AssertionError(
        f"Expected 40 holdout rows, found {len(holdout)}."
    )
if len(external) != 119:
    raise AssertionError(
        f"Expected 119 external rows, found {len(external)}."
    )

labelled_columns = [
    "event_date",
    "sample_block",
    "decision_rule",
    "decision_rule_order",
    "decision_cutoff_utc",
    "development_fold",
    "forecast_daily_max_c",
    "hko_daily_max_c",
    "residual_c",
    "day_index",
    "forecast_lead_hours",
    "run_initialisation_hour_utc",
    "day_of_year_sin",
    "day_of_year_cos",
    "selected_run_key",
    "selected_run_initialisation_utc",
    "selected_run_available_utc",
    "n_admissible_same_rule_residuals",
    "current_label_available_utc",
]

blind_columns = [
    column
    for column in labelled_columns
    if column
    not in {
        "hko_daily_max_c",
        "residual_c",
        "current_label_available_utc",
    }
] + [
    "evaluation_block",
    "row_role",
]

development_panel = development[
    labelled_columns
].sort_values(
    [
        "event_date",
        "decision_rule_order",
    ]
).reset_index(drop=True)
development_panel[
    "date_balanced_development_weight"
] = date_balanced_weights(
    development_panel,
    "event_date",
)

blind_evaluation_panel = pd.concat(
    [
        holdout[blind_columns],
        external[blind_columns],
    ],
    ignore_index=True,
).sort_values(
    [
        "event_date",
        "decision_rule_order",
    ]
).reset_index(drop=True)

forbidden_blind = {
    "hko_daily_max_c",
    "residual_c",
    "forecast_error_c",
    "Y_event_int",
    "Y_no_int",
    "p_market",
    "current_label_available_utc",
}

present = forbidden_blind.intersection(
    blind_evaluation_panel.columns
)
if present:
    raise AssertionError(
        "Blind evaluation panel contains forbidden outcome "
        f"fields: {sorted(present)}"
    )

if blind_evaluation_panel[
    [
        "forecast_daily_max_c",
        "day_index",
        "forecast_lead_hours",
        "run_initialisation_hour_utc",
        "day_of_year_sin",
        "day_of_year_cos",
    ]
].isna().any().any():
    raise AssertionError(
        "A blind evaluation row lacks a required feature."
    )

print("Canonical current feature panels: PASS")
print(f"Development labelled rows: {len(development_panel):,}")
print(
    f"Blind holdout/external rows: "
    f"{len(blind_evaluation_panel):,}"
)

Canonical current feature panels: PASS
Development labelled rows: 144
Blind holdout/external rows: 159


In [4]:
candidate_registry = pd.DataFrame(CANDIDATE_ROWS)
feature_registry = pd.DataFrame(FEATURE_ROWS)
quantile_grid = pd.DataFrame(
    {
        "quantile_level": QUANTILE_LEVELS,
        "residual_quantile_column": [
            f"residual_q{int(round(level * 100)):02d}"
            for level in QUANTILE_LEVELS
        ],
        "hko_quantile_column": [
            f"hko_q{int(round(level * 100)):02d}"
            for level in QUANTILE_LEVELS
        ],
    }
)

candidate_registry["candidate_order"] = np.arange(
    1,
    len(candidate_registry) + 1,
)
candidate_registry[
    "uses_market_information"
] = False
candidate_registry[
    "uses_artificial_ensemble_features"
] = False
candidate_registry[
    "uses_fixed_gaussian_bridge"
] = False
candidate_registry[
    "development_only_selection"
] = True
candidate_registry[
    "holdout_or_external_used_for_selection"
] = False

if len(candidate_registry) != 9:
    raise AssertionError(
        f"Expected nine candidates, "
        f"found {len(candidate_registry)}."
    )
if len(quantile_grid) != 99:
    raise AssertionError(
        f"Expected 99 quantiles, found {len(quantile_grid)}."
    )

if not np.allclose(
    quantile_grid["quantile_level"],
    np.arange(0.01, 1.00, 0.01),
    rtol=0.0,
    atol=1e-12,
):
    raise AssertionError(
        "Unexpected predictive quantile grid."
    )

if candidate_registry[
    [
        "uses_market_information",
        "uses_artificial_ensemble_features",
        "uses_fixed_gaussian_bridge",
        "holdout_or_external_used_for_selection",
    ]
].any().any():
    raise AssertionError(
        "A prohibited candidate-design flag is true."
    )

print("Candidate and feature registries: PASS")
display(candidate_registry)
display(feature_registry)

Candidate and feature registries: PASS


,candidate_id,model_family,scope_type,scope_id_pattern,distribution_type,complexity_rank,principal_family_comparison,candidate_order,uses_market_information,uses_artificial_ensemble_features,uses_fixed_gaussian_bridge,development_only_selection,holdout_or_external_used_for_selection
0,raw_deterministic,RAW,POOLED,pooled_all_rules,DEGENERATE,1,False,1,False,False,False,True,False
1,pooled_mean_residual,BASELINE,POOLED,pooled_all_rules,DEGENERATE,2,False,2,False,False,False,True,False
2,rule_mean_residual,BASELINE,RULE_SPECIFIC,rule_specific_{decision_rule},DEGENERATE,3,False,3,False,False,False,True,False
3,pooled_empirical_residual,BASELINE,POOLED,pooled_all_rules,EMPIRICAL_QUANTILES,4,False,4,False,False,False,True,False
4,rule_empirical_residual,BASELINE,RULE_SPECIFIC,rule_specific_{decision_rule},EMPIRICAL_QUANTILES,5,False,5,False,False,False,True,False
5,gp_rbf_rule,GAUSSIAN_PROCESS,RULE_SPECIFIC,rule_specific_{decision_rule},GAUSSIAN_QUANTILES,6,True,6,False,False,False,True,False
6,gp_matern32_rule,GAUSSIAN_PROCESS,RULE_SPECIFIC,rule_specific_{decision_rule},GAUSSIAN_QUANTILES,7,True,7,False,False,False,True,False
7,catboost_quantile_pooled,TREE,POOLED,pooled_all_rules,INTERPOLATED_QUANTILES,8,True,8,False,False,False,True,False
8,catboost_quantile_rule,TREE,RULE_SPECIFIC,rule_specific_{decision_rule},INTERPOLATED_QUANTILES,9,True,9,False,False,False,True,False


,feature_name,gp_rule_specific,tree_pooled,tree_rule_specific,description
0,day_index,True,True,True,Calendar days since 16 March 2026.
1,forecast_daily_max_c,False,True,True,Deterministic ECMWF forecast daily maximum.
2,decision_rule_order,False,True,False,Ordinal decision-rule indicator for pooled trees.
3,forecast_lead_hours,False,True,True,Hours from selected-run availability to decisi...
4,run_initialisation_hour_utc,False,True,True,UTC cycle hour of the selected deterministic run.
5,day_of_year_sin,False,True,True,Seasonal sine coordinate.
6,day_of_year_cos,False,True,True,Seasonal cosine coordinate.


In [5]:
check_rows: list[dict[str, Any]] = []

def add_check(
    check: str,
    passed: bool,
    detail: str,
) -> None:
    check_rows.append(
        {
            "check": check,
            "passed": bool(passed),
            "detail": detail,
            "blocking": True,
        }
    )

add_check(
    "verified_upstream_releases",
    all(
        summary.get("verdict") == "PASS"
        for summary in summaries.values()
    ),
    "18s, 18uA and 18uB are PASS",
)
add_check(
    "candidate_models_9",
    len(candidate_registry) == 9,
    f"candidates={len(candidate_registry)}",
)
add_check(
    "quantile_grid_99",
    len(quantile_grid) == 99,
    "levels 0.01 through 0.99",
)
add_check(
    "development_labelled_rows_144",
    len(development_panel) == 144,
    "development model-ready date-rule rows",
)
add_check(
    "blind_evaluation_rows_159",
    len(blind_evaluation_panel) == 159,
    "40 holdout plus 119 external rows",
)
add_check(
    "blind_evaluation_has_no_outcomes",
    not bool(
        forbidden_blind.intersection(
            blind_evaluation_panel.columns
        )
    ),
    "no realised HKO, residual, event or market field",
)
add_check(
    "market_features_excluded",
    not candidate_registry[
        "uses_market_information"
    ].any(),
    "weather residual modelling only",
)
add_check(
    "artificial_ensemble_features_excluded",
    not candidate_registry[
        "uses_artificial_ensemble_features"
    ].any(),
    "no proxy ensemble inputs",
)
add_check(
    "fixed_gaussian_bridge_excluded",
    not candidate_registry[
        "uses_fixed_gaussian_bridge"
    ].any(),
    "predictive distributions are data-driven",
)
add_check(
    "selection_protocol_development_only",
    (
        SELECTION_PROTOCOL["holdout_used"] is False
        and SELECTION_PROTOCOL[
            "external_test_used"
        ]
        is False
    ),
    "selection boundary frozen",
)

integrity = pd.DataFrame(check_rows)
if not integrity["passed"].all():
    raise AssertionError(
        "18vA blocking checks failed:\n"
        + integrity.loc[
            ~integrity["passed"]
        ].to_string(index=False)
    )

issues = pd.DataFrame(
    columns=[
        "issue_level",
        "issue_code",
        "event_date",
        "decision_rule",
        "candidate_id",
        "detail",
        "blocking",
    ]
)

print("18vA integrity checks: PASS")

18vA integrity checks: PASS


In [6]:
output_frames = {
    "candidate_registry": candidate_registry,
    "feature_registry": feature_registry,
    "quantile_grid": quantile_grid,
    "development_panel": development_panel,
    "blind_evaluation_panel": blind_evaluation_panel,
    "integrity": integrity,
    "issues": issues,
}

output_paths = {
    "candidate_registry": (
        OUT_DIR / "18vA_candidate_registry.csv"
    ),
    "feature_registry": (
        OUT_DIR / "18vA_feature_registry.csv"
    ),
    "quantile_grid": (
        OUT_DIR / "18vA_quantile_grid.csv"
    ),
    "development_panel": (
        OUT_DIR
        / "18vA_development_labelled_feature_panel.csv"
    ),
    "blind_evaluation_panel": (
        OUT_DIR
        / "18vA_blind_evaluation_feature_panel.csv"
    ),
    "integrity": (
        OUT_DIR / "18vA_integrity_checks.csv"
    ),
    "issues": OUT_DIR / "18vA_issues.csv",
}

for key, frame in output_frames.items():
    output = frame.copy()

    for column in output.columns:
        if "date" in column.lower():
            if pd.api.types.is_datetime64_any_dtype(
                output[column]
            ):
                output[column] = output[
                    column
                ].dt.strftime("%Y-%m-%d")

        if (
            "cutoff" in column.lower()
            or column.lower().endswith("_utc")
            or "available" in column.lower()
            or "initialisation" in column.lower()
        ):
            output[column] = output[column].astype(
                "string"
            )

    output.to_csv(
        output_paths[key],
        index=False,
    )

protocol = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": "PASS",
    "residual_definition": (
        "HKO daily maximum minus deterministic forecast maximum"
    ),
    "candidate_models": CANDIDATE_ROWS,
    "gp_rule_specific_features": ["day_index"],
    "tree_pooled_features": feature_registry.loc[
        feature_registry["tree_pooled"],
        "feature_name",
    ].tolist(),
    "tree_rule_specific_features": feature_registry.loc[
        feature_registry["tree_rule_specific"],
        "feature_name",
    ].tolist(),
    "predictive_quantile_levels": (
        QUANTILE_LEVELS.tolist()
    ),
    "tree_training_quantiles": (
        TREE_TRAINING_QUANTILES
    ),
    "catboost_fixed_hyperparameters": {
        "iterations": 250,
        "depth": 3,
        "learning_rate": 0.03,
        "l2_leaf_reg": 8.0,
        "random_seed": 20260721,
        "bootstrap_type": "No",
        "random_strength": 0.0,
        "thread_count": 1,
    },
    "gaussian_process_candidates": {
        "gp_rbf_rule": (
            "ConstantKernel times RBF plus WhiteKernel"
        ),
        "gp_matern32_rule": (
            "ConstantKernel times Matern(nu=1.5) "
            "plus WhiteKernel"
        ),
        "input_dimension": 1,
        "input_feature": "day_index",
        "normalise_target": True,
        "optimizer_restarts": 0,
    },
    "selection_protocol": SELECTION_PROTOCOL,
    "market_information_used": False,
    "artificial_ensemble_features_used": False,
    "fixed_gaussian_bridge_used": False,
    "holdout_or_external_outcomes_exported": False,
}

protocol_path = OUT_DIR / "18vA_protocol.json"
protocol_path.write_text(
    json.dumps(
        protocol,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

source_inventory = pd.DataFrame(
    [
        {
            "input_role": "18s_weather_panel",
            "path": str(WEATHER_PATH.relative_to(ROOT)),
            "rows": len(weather),
            "sha256": sha256_file(WEATHER_PATH),
        },
        {
            "input_role": "18uA_date_rule_assignment",
            "path": str(
                ASSIGNMENT_PATH.relative_to(ROOT)
            ),
            "rows": len(assignment),
            "sha256": sha256_file(ASSIGNMENT_PATH),
        },
        {
            "input_role": "18uB_scope_registry",
            "path": str(SCOPE_PATH.relative_to(ROOT)),
            "rows": len(scopes),
            "sha256": sha256_file(SCOPE_PATH),
        },
        {
            "input_role": "18uB_fold_training_membership",
            "path": str(
                FOLD_TRAIN_PATH.relative_to(ROOT)
            ),
            "rows": len(fold_train),
            "sha256": sha256_file(FOLD_TRAIN_PATH),
        },
        {
            "input_role": "18uB_fold_validation_membership",
            "path": str(
                FOLD_VALIDATION_PATH.relative_to(ROOT)
            ),
            "rows": len(fold_validation),
            "sha256": sha256_file(
                FOLD_VALIDATION_PATH
            ),
        },
        {
            "input_role": "18uB_final_fit_membership",
            "path": str(
                FINAL_FIT_PATH.relative_to(ROOT)
            ),
            "rows": len(final_fit),
            "sha256": sha256_file(FINAL_FIT_PATH),
        },
        {
            "input_role": "18uB_evaluation_support",
            "path": str(
                EVALUATION_PATH.relative_to(ROOT)
            ),
            "rows": len(evaluation),
            "sha256": sha256_file(EVALUATION_PATH),
        },
    ]
)
source_inventory_path = (
    OUT_DIR / "18vA_source_inventory.csv"
)
source_inventory.to_csv(
    source_inventory_path,
    index=False,
)

summary = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": "PASS",
    "candidate_models": int(len(candidate_registry)),
    "baseline_candidates": int(
        candidate_registry[
            "model_family"
        ].isin(["RAW", "BASELINE"]).sum()
    ),
    "gaussian_process_candidates": int(
        candidate_registry[
            "model_family"
        ].eq("GAUSSIAN_PROCESS").sum()
    ),
    "tree_candidates": int(
        candidate_registry[
            "model_family"
        ].eq("TREE").sum()
    ),
    "predictive_quantile_grid_points": int(
        len(quantile_grid)
    ),
    "development_labelled_rows": int(
        len(development_panel)
    ),
    "blind_holdout_rows": int(
        blind_evaluation_panel[
            "evaluation_block"
        ].eq("INTERNAL_HOLDOUT").sum()
    ),
    "blind_external_rows": int(
        blind_evaluation_panel[
            "evaluation_block"
        ].eq("EXTERNAL_TEST").sum()
    ),
    "market_information_used": False,
    "artificial_ensemble_features_used": False,
    "fixed_gaussian_bridge_used": False,
    "holdout_or_external_outcomes_exported": False,
    "issue_rows": 0,
    "integrity_checks_passed": int(
        integrity["passed"].sum()
    ),
    "integrity_checks_total": int(len(integrity)),
}

summary_path = OUT_DIR / "18vA_summary.json"
summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

environment = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "revision": "v1",
}
environment_path = OUT_DIR / "18vA_environment.json"
environment_path.write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

report_lines = [
    "# 18vA residual-model design and features",
    "",
    "**PASS**",
    "",
    "## Candidate set",
    "",
    "| Candidate | Family | Scope | Distribution |",
    "|---|---|---|---|",
]

for row in candidate_registry.itertuples(index=False):
    report_lines.append(
        f"| {row.candidate_id} | "
        f"{row.model_family} | "
        f"{row.scope_type} | "
        f"{row.distribution_type} |"
    )

report_lines.extend(
    [
        "",
        "## Frozen panels",
        "",
        (
            f"- Development labelled date-rule rows: "
            f"{len(development_panel):,}"
        ),
        (
            f"- Blind internal-holdout rows: "
            f"{blind_evaluation_panel['evaluation_block'].eq('INTERNAL_HOLDOUT').sum():,}"
        ),
        (
            f"- Blind external rows: "
            f"{blind_evaluation_panel['evaluation_block'].eq('EXTERNAL_TEST').sum():,}"
        ),
        "",
        "## Selection hierarchy",
        "",
        (
            "Primary: date-balanced mean CRPS on the common "
            "freeze-admissible development OOF support."
        ),
        (
            "Secondary: date-balanced mean absolute error. "
            "Tertiary: lower pre-frozen complexity rank."
        ),
        "",
        "## Exclusions",
        "",
        (
            "No market, artificial ensemble or fixed Gaussian "
            "bridge feature is used. Holdout and external outcomes "
            "are absent from the blind feature panel."
        ),
    ]
)

report_path = (
    REPORT_DIR
    / "18vA_residual_model_design_and_features_report.md"
)
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

manifest_rows = []
for root in [OUT_DIR, REPORT_DIR]:
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "18vA_sha256_manifest.csv":
            continue

        manifest_rows.append(
            {
                "path": str(path.relative_to(ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

manifest_path = OUT_DIR / "18vA_sha256_manifest.csv"
pd.DataFrame(manifest_rows).to_csv(
    manifest_path,
    index=False,
)

print(json.dumps(summary, indent=2))
print("18vA residual-model design release: PASS")

{
  "step": "18vA",
  "generated_at_utc": "2026-07-21T22:42:21.844979+00:00",
  "verdict": "PASS",
  "candidate_models": 9,
  "baseline_candidates": 5,
  "gaussian_process_candidates": 2,
  "tree_candidates": 2,
  "predictive_quantile_grid_points": 99,
  "development_labelled_rows": 144,
  "blind_holdout_rows": 40,
  "blind_external_rows": 119,
  "market_information_used": false,
  "artificial_ensemble_features_used": false,
  "fixed_gaussian_bridge_used": false,
  "holdout_or_external_outcomes_exported": false,
  "issue_rows": 0,
  "integrity_checks_passed": 10,
  "integrity_checks_total": 10
}
18vA residual-model design release: PASS
